In [32]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tqdm import tqdm
import numpy as np
import base64
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v3 as iio
import io
import base64
import tempfile
from PIL import Image, ImageDraw
from IPython.display import HTML, display, Video
from typing import List, Tuple, Optional
import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
ROOT = Path("/work/cs-503/santanto/waymo/")
VAL_CHUNKS = sorted(ROOT.glob("validation__*"))

CAMERA_ID_COL = "camera_name"
IMAGE_JPEG_COL = "image_jpeg"
BBOX_COLS = ["bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]

In [34]:
def display_results(
    image_jpegs: List[bytes], 
    bboxes: Optional[List[List[Tuple[float, float, float, float]]]] = None,
    fps: int = 10,
    output_path: Optional[Path] = None,
):
    """
    Plot video of images with bounding boxes in the notebook.
    Args:
        image_jpegs: List of JPEG bytes to display as video frames.
        bboxes: List of bounding boxes per image, each as (xmin, ymin, xmax, ymax).
        fps: Frames per second for the video.
        output_path: If provided, saves the video to this path. Defaults to a temp file for display.
    """
    image_jpegs = image_jpegs[::15]

    if bboxes is None:
        bboxes = [[] for _ in image_jpegs]

    buffer = []
    for img_jpeg, img_bboxes in tqdm(zip(image_jpegs, bboxes), total=len(image_jpegs)):
        img = Image.open(io.BytesIO(img_jpeg)).resize((1920 // 2, 1280 // 2))

        if img_bboxes:
            draw = ImageDraw.Draw(img)
            for bbox in img_bboxes:
                xmin, ymin, xmax, ymax = bbox
                draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=2)

        buffer.append(img)

    if not buffer:
        return

    save_path = Path(output_path) if output_path else Path(tempfile.NamedTemporaryFile(suffix=".gif").name)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    buffer[0].save(
        save_path,
        save_all=True,
        append_images=buffer[1:],
        duration=1000 // fps,
        loop=0,
    )

    if output_path is None:
        with open(save_path, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        display(HTML(f'<img src="data:image/gif;base64,{b64}"/>'))

In [35]:
SCENE_OF_INTEREST = [
    "validation__8079607115087394458_1240_000_1260_000",  # Intersection, but only one interesting car
    "validation__1906113358876584689_1359_560_1379_560",  # Easy street, with pedestrian crossing
    "validation__11450298750351730790_1431_750_1451_750", # Intersection, same direction
    "validation__2506799708748258165_6455_000_6475_000",  # Highway, many lane changes
    "validation__14300007604205869133_1160_000_1180_000", # Night, many object, bike direction change
    "validation__15224741240438106736_960_000_980_000",   # Many objects, lane changes
    "validation__3731719923709458059_1540_000_1560_000",  # Car coming from thr front, turning just before
    "validation__18446264979321894359_3700_000_3720_000", # Similar but following the car
    "validation__8956556778987472864_3404_790_3424_790",  # Pedestrian comes out at the last second
]

for scene in SCENE_OF_INTEREST:
    print(f"Processing scene: {scene}")
    chunk_path = ROOT / scene
    out_path = Path(f"/home/gromb/project/outputs/interesting_scenes/{scene}.gif")
    if out_path.exists():
        print(f"Output already exists at {out_path}, skipping display.")
        continue
    print(f"Reading data from: {chunk_path}")
    df = pd.read_parquet(chunk_path / "images.parquet")
    image_jpegs = df[IMAGE_JPEG_COL].tolist()
    print(f"Displaying results for scene: {scene}")
    display_results(image_jpegs, fps=10, output_path=out_path)
    print(f"Finished processing scene: {scene}\n")

Processing scene: validation__8079607115087394458_1240_000_1260_000
Output already exists at /home/gromb/project/outputs/interesting_scenes/validation__8079607115087394458_1240_000_1260_000.gif, skipping display.
Processing scene: validation__1906113358876584689_1359_560_1379_560
Output already exists at /home/gromb/project/outputs/interesting_scenes/validation__1906113358876584689_1359_560_1379_560.gif, skipping display.
Processing scene: validation__11450298750351730790_1431_750_1451_750
Output already exists at /home/gromb/project/outputs/interesting_scenes/validation__11450298750351730790_1431_750_1451_750.gif, skipping display.
Processing scene: validation__2506799708748258165_6455_000_6475_000
Output already exists at /home/gromb/project/outputs/interesting_scenes/validation__2506799708748258165_6455_000_6475_000.gif, skipping display.
Processing scene: validation__14300007604205869133_1160_000_1180_000
Output already exists at /home/gromb/project/outputs/interesting_scenes/valid

In [ ]:
# Sample 10 random scenes from the validation set
sampled_scenes = np.random.choice(VAL_CHUNKS, size=10, replace=False)

for scene_path in tqdm(sampled_scenes, desc="Processing scenes"):
    df = pd.read_parquet(scene_path / "images.parquet")
    df = df[df[CAMERA_ID_COL] == 1]

    # Print distinct scene IDs in the current chunk
    scene_ids = df["scene_id"].unique()
    print(f"Scene IDs in {scene_path.name}: {scene_ids}")

    # Display the scene
    image_jpegs = df[IMAGE_JPEG_COL].tolist()
    display_results(image_jpegs, fps=10, output_path=f"/home/gromb/project/outputs/sample_scenes/{scene_path.stem}.gif")

Processing scenes:   0%|          | 0/10 [00:00<?, ?it/s]

Scene IDs in validation__7932945205197754811_780_000_800_000: <ArrowStringArray>
['7932945205197754811_780_000_800_000']
Length: 1, dtype: str


Processing scenes:  10%|█         | 1/10 [00:07<01:03,  7.01s/it]

Scene IDs in validation__7119831293178745002_1094_720_1114_720: <ArrowStringArray>
['7119831293178745002_1094_720_1114_720']
Length: 1, dtype: str


Processing scenes:  20%|██        | 2/10 [00:13<00:52,  6.57s/it]

Scene IDs in validation__346889320598157350_798_187_818_187: <ArrowStringArray>
['346889320598157350_798_187_818_187']
Length: 1, dtype: str


Processing scenes:  30%|███       | 3/10 [00:18<00:43,  6.17s/it]

Scene IDs in validation__14333744981238305769_5658_260_5678_260: <ArrowStringArray>
['14333744981238305769_5658_260_5678_260']
Length: 1, dtype: str
